# Auditable Wikipedia RAG-ETL Pipeline — Full Offline Build

## Project Overview

This notebook demonstrates the **full offline pipeline** for building an auditable Wikipedia-based FAISS IVF retrieval system.

### System Architecture

The pipeline consists of the following stages:

1. **INGEST**: Read raw Wikipedia JSONL documents into `raw_documents` table
2. **CLEAN**: Validate and normalize text; track rejections in `rejected_documents`
3. **CURATE**: SCD (Slowly Changing Dimension) logic to track new/updated/unchanged documents in `clean_documents`
4. **CHUNK**: Sliding-window tokenization (384 tokens, 320 stride) into `fact_chunks`
5. **EMBED**: E5-base-v2 embeddings stored as per-run `.npy` files; metadata in `fact_embeddings`
6. **INDEX**: FAISS IVFFlat index built from all active embeddings; registry in `faiss_index_registry`
7. **RECONCILIATION**: Count-based sanity checks across all tables
8. **AUDIT**: Cross-run idempotency and consistency checks

### Design Decisions
- **Embedding storage**: Per-run numpy files — `outputs/full/embeddings/run_{run_id}.npy`
- **FAISS vector map**: `outputs/full/faiss/vector_map.npy` maps FAISS IDs to `chunk_id`s
- **DuckDB**: All metadata in `outputs/full/wiki.duckdb`
- **E5 prefix**: `passage:` for indexing, `query:` for retrieval
- **IVF adaptive nlist**: `min(100, max(4, N // 10))`

## Setup

In [ ]:
import os
import subprocess

# Navigate to project root so all relative paths work correctly
os.chdir('..')
print(f"Working directory: {os.getcwd()}")

import pandas as pd
import duckdb

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
print("Setup complete.")

## Raw Data Preview

Preview the first 5 rows of the raw Wikipedia JSONL file.

In [ ]:
raw_path = 'data/full/raw_wiki.jsonl'

if os.path.exists(raw_path):
    df_raw = pd.read_json(raw_path, lines=True, nrows=5)
    print(f"Columns: {list(df_raw.columns)}")
    print(f"Shape: {df_raw.shape}")
    display(df_raw[['id', 'title', 'url']].head())
    print("\nSample text (first doc):")
    print(str(df_raw['text'].iloc[0])[:300] + "...")
else:
    print(f"File not found: {raw_path}")
    print("Please download Wikipedia data first.")
    print("See README.md for instructions.")

## Full Rebuild

Run the complete ETL pipeline in **full** mode. This processes all documents through the SCD logic.

- Unchanged documents (same id + content hash) are skipped for chunking/embedding
- This makes the pipeline **idempotent**

In [ ]:
result = subprocess.run(
    [
        'python', 'src/build.py',
        '--mode', 'full',
        '--dataset-name', 'full',
        '--input', 'data/full/raw_wiki.jsonl',
        '--output', 'outputs/full',
        '--db', 'outputs/full/wiki.duckdb',
        '--index-type', 'ivf'
    ],
    check=True,
    capture_output=False
)
print(f"\nReturn code: {result.returncode}")

## DuckDB Tables Overview

In [ ]:
conn = duckdb.connect('outputs/full/wiki.duckdb', read_only=True)

tables = conn.execute("SHOW TABLES").df()
print("All tables in wiki.duckdb:")
display(tables)

for tbl in tables['name'].tolist():
    count = conn.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"  {tbl}: {count} rows")

## Runs Table

In [ ]:
df_runs = conn.execute("SELECT * FROM runs").df()
print(f"Total runs: {len(df_runs)}")
display(df_runs)

## Row Count Reconciliation

Checks that active chunks == active embeddings == index vectors.

In [ ]:
df_recon = conn.execute("SELECT * FROM row_count_reconciliation").df()
print("Row count reconciliation:")
display(df_recon)

## FAISS IVF Index Registry

In [ ]:
df_faiss = conn.execute("SELECT * FROM faiss_index_registry").df()
print("FAISS index registry:")
display(df_faiss)

## Latency Logs

Stage-by-stage timing breakdown.

In [ ]:
df_latency = conn.execute(
    "SELECT stage_name, duration_seconds, row_count FROM latency_logs ORDER BY start_time"
).df()
print("Latency logs by stage:")
display(df_latency)

total = df_latency['duration_seconds'].sum()
print(f"\nTotal pipeline time: {total:.1f}s")

## Retrieval Examples

Run example queries against the full index.

In [ ]:
from src.retrieve import retrieve

DB_PATH = 'outputs/full/wiki.duckdb'
INDEX_PATH = 'outputs/full/faiss/index.faiss'

queries = [
    "What is machine learning?",
    "History of the Roman Empire",
    "Climate change and global warming"
]

for q in queries:
    print(f"\n{'='*60}")
    print(f"Query: {q}")
    print('='*60)
    df_result = retrieve(q, DB_PATH, INDEX_PATH, top_k=3)
    if df_result.empty:
        print("No results found.")
    else:
        for _, row in df_result.iterrows():
            print(f"[{row['rank']}] Score={row['score']:.4f} | {row['title']}")
            print(f"     {str(row['chunk_text'])[:150]}...")

## Retrieval Eval Table

In [ ]:
df_eval = conn.execute("SELECT * FROM retrieval_eval").df()
print(f"Retrieval eval records: {len(df_eval)}")
if not df_eval.empty:
    display(df_eval)
else:
    print("No retrieval eval records yet. Add labeled queries to populate this table.")

In [ ]:
conn.close()
print("Connection closed.")

## Summary

### System Design Summary

This pipeline implements a fully auditable Wikipedia RAG-ETL system:

**Data Grain:**
- `raw_documents`: one row per ingested Wikipedia document (per run)
- `clean_documents`: one active row per document (SCD Type 2 versioning by run_id)
- `fact_chunks`: one row per sliding-window text chunk (SCD Type 2)
- `fact_embeddings`: one row per embedded chunk (run-level)

**Update Contract:**
1. NEW docs → inserted into clean_documents, chunked, embedded
2. UPDATED docs → old rows deactivated, new rows inserted with updated content
3. UNCHANGED docs → skipped (idempotent)
4. REJECTED docs → tracked in rejected_documents with reason code

**FAISS IVF Design:**
- IndexIVFFlat with INNER_PRODUCT metric (cosine similarity via normalized vectors)
- Adaptive nlist: `min(100, max(4, N // 10))`
- nprobe=10 for search
- Full rebuild from all active embeddings on every run

**Monitoring:**
- `runs` table: per-run metadata and counts
- `latency_logs`: per-stage timing
- `row_count_reconciliation`: cross-table count checks
- `audit_results`: cross-run idempotency verification